Load dataset

In [3]:
from transformers import AutoProcessor

# --- CUSTOMIZATION SECTION ---
config = {
    # Emotion mapping: you can unify labels across datasets here
    "label_map": {
        "neutral": 0,
        "happy": 1,
        "sad": 2,
        "angry": 3,
        "fear": 4,
        "disgust": 5,
        "surprise": 6
    },

    "model_id": "facebook/w2v-bert-2.0",
    "sampling_rate": 16000
}



In [6]:
from datasets import load_dataset, concatenate_datasets, Audio, get_dataset_split_names

# 1. Fetch dataset metadata to automatically extract all available configurations (languages/sub-datasets)
configs = get_dataset_split_names("amu-cai/CAMEO")
print(f"Found {len(configs)} subsets: {configs}")

# 2. Load all subsets using a loop (List Comprehension)
print("Loading and merging data... This may take a moment.")
datasets_list = [
    load_dataset("amu-cai/CAMEO", split=conf)
    for conf in configs
]

full_dataset = concatenate_datasets(datasets_list)
print(full_dataset[0])
full_dataset = full_dataset.class_encode_column("emotion")


Found 13 subsets: ['crema_d', 'cafe', 'emns', 'emozionalmente', 'enterface', 'jl_corpus', 'mesd', 'nemo', 'oreau', 'pavoque', 'ravdess', 'resd', 'subesco']
Loading and merging data... This may take a moment.
{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x00000206C0F69430>, 'emotion': 'anger', 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}
{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x0000020570C65670>, 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}


In [9]:
print(full_dataset.features["emotion"])
print(full_dataset.features["emotion"].str2int("anger"))
print(full_dataset.features["emotion"].int2str(0))


ClassLabel(names=['anger', 'anxiety', 'apology', 'assertiveness', 'calm', 'concern', 'disgust', 'encouragement', 'enthusiasm', 'excitement', 'fear', 'happiness', 'neutral', 'poker', 'sadness', 'sarcasm', 'surprise'])
0
anger


In [11]:
# 2. LABEL PREPARATION AND SAMPLING
def preprocess_labels(example):
    example["label"] = example["emotion"]
    return example


# Apply label mapping and remove samples not in mapping (label -1)
full_dataset = full_dataset.map(preprocess_labels)

# Force resampling to 16kHz
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=config["sampling_rate"]))

# 3. PROCESSOR AND MODEL INPUT PREPARATION
processor = AutoProcessor.from_pretrained(config["model_id"])

def prepare_dataset(batch):
    audio = batch["audio"]

    # Convert audio signal into model input (Wav2Vec2-BERT)
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    )

    batch["input_features"] = inputs.input_features[0]

    # Attention mask may also be required
    if "attention_mask" in inputs:
        batch["attention_mask"] = inputs.attention_mask[0]

    return batch

# 4. FINAL PROCESSING (Mapping to model input)
# Remove old columns, keep only what Trainer needs
encoded_dataset = full_dataset.map(
    prepare_dataset,
    remove_columns=full_dataset.column_names,
    num_proc=4  # Speed up with multiprocessing
)

# Train/validation split
final_splits = encoded_dataset.train_test_split(test_size=0.15, seed=42)

print("\nDataset summary:")
print(f"Total number of samples: {len(encoded_dataset)}")
print(f"Split: {final_splits}")

Map: 100%|██████████| 41265/41265 [00:05<00:00, 7332.51 examples/s]
C:\Users\witze\Documents\GitHub\wav2vec2-xls-r-300m-emotion-detection\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\witze\.cache\huggingface\hub\models--facebook--w2v-bert-2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ImportError: 
 requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
from transformers import AutoModelForAudioClassification

id2label = {0: "happiness", 1: "anger", 2: "fear", 3: "disgust", 5: "sadness", 6: "neutral"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    label2id=label2id,
    id2label=id2label
)

# DOBRA PRAKTYKA: Zamrożenie warstw wyciągających cechy (feature extractor).
# Wav2Vec2-BERT to duży model. Zamrożenie dolnych warstw zaoszczędzi mnóstwo RAMu
# na karcie graficznej i zapobiegnie zapominaniu (catastrophic forgetting).
model.freeze_feature_encoder()

In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

# Parametry treningu
training_args = TrainingArguments(
    output_dir="./w2v-bert-emotion",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5, # Niski learning rate dla fine-tuningu!
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, # Pomaga, jeśli masz mało pamięci VRAM
    num_train_epochs=5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=processor, # w modelach audio procesor pełni rolę tokenizera
    compute_metrics=compute_metrics,
)

# Start treningu!
trainer.train()